<a href="https://colab.research.google.com/github/nanafish/ORS/blob/main/%E4%BC%9A%E5%A0%B4%E5%90%88%E8%A8%88%E5%A5%91%E7%B4%84%E7%8E%87%E3%83%BB%E9%87%91%E9%A1%8D%E3%83%BB%E5%8D%98%E4%BE%A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title
# ============================================================
# 予約特典別：来場率・契約率・購入総額・平均単価 分析
# 対象者全体 vs セグメント層（会場は統合）
# Google Colab 1セル実行用
# ============================================================

import os
import re
import sys
import math
import glob
import json
import shutil
import hashlib
import zipfile
import unicodedata
import subprocess
from pathlib import Path
from datetime import datetime

# ---------- 必要ライブラリ ----------
def ensure_package(import_name, pip_name=None):
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

ensure_package("openpyxl")
ensure_package("xlsxwriter")
ensure_package("seaborn")
ensure_package("japanize_matplotlib", "japanize-matplotlib")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import japanize_matplotlib
from google.colab import files

# ============================================================
# 設定：通常は INPUT_MODE だけ確認してください
# ============================================================
INPUT_MODE = "upload"       # "upload" または "drive"
DRIVE_FOLDER = "/content/drive/MyDrive/予約特典分析入力"
SEARCH_SUBFOLDERS = True
AUTO_DOWNLOAD_ZIP = True

# 集計条件
TARGET_VALUE = "対象"
SEGMENT_VALUE = "セグメント層"
UNKNOWN_BENEFIT = "（特典不明）"

# 出力先
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path(f"/content/予約特典別分析_{stamp}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 共通関数
# ============================================================
def norm_text(v):
    """比較用文字列：全半角統一・改行/空白除去。"""
    if pd.isna(v):
        return ""
    s = unicodedata.normalize("NFKC", str(v))
    return re.sub(r"[\s\u3000]+", "", s).strip()


def norm_col(v):
    """列名照合用。"""
    s = norm_text(v)
    return re.sub(r"[_\-‐–—・/／\\（）()［］\[\]【】:：]", "", s).lower()


def is_nonblank(series):
    """日付がExcelシリアル値でも文字列でも、空欄でなければTrue。"""
    s = series.copy()
    return s.notna() & s.astype(str).str.strip().ne("") & ~s.astype(str).str.lower().isin(["nan", "none", "nat"])


def to_number(series, default=0):
    """通貨記号・カンマ等を除いて数値化。"""
    cleaned = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("¥", "", regex=False)
        .str.replace("￥", "", regex=False)
        .str.replace("円", "", regex=False)
        .str.replace(r"[^0-9.\-]", "", regex=True)
    )
    return pd.to_numeric(cleaned, errors="coerce").fillna(default)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def safe_div(numer, denom):
    numer = np.asarray(numer, dtype=float)
    denom = np.asarray(denom, dtype=float)
    return np.divide(numer, denom, out=np.full_like(numer, np.nan), where=denom != 0)


def yen_label(v):
    if pd.isna(v):
        return "—"
    return f"¥{v:,.0f}"


def rate_label(v):
    if pd.isna(v):
        return "—"
    return f"{v:.1%}"


def yen_axis(v, _):
    av = abs(v)
    if av >= 100_000_000:
        return f"{v/100_000_000:.1f}億"
    if av >= 10_000:
        return f"{v/10_000:.0f}万"
    return f"{v:,.0f}"


COLUMN_ALIASES = {
    "匿名予約ID": ["匿名予約ID", "予約ID", "匿名ID"],
    "会場名": ["会場名", "会場"],
    "予約者元ファイル": ["予約者元ファイル", "予約元ファイル", "元ファイル"],
    "来場日": ["来場日", "来展日", "来場日時", "来展日時"],
    "契約日": ["契約日", "購入日", "成約日"],
    "セグメント区分": ["セグメント区分", "セグメント", "セグメント層区分"],
    "予約特典コード": ["予約特典コード", "特典コード", "予約特典cd"],
    "予約特典名": ["予約特典名", "ご希望の予約特典", "特典名", "予約特典"],
    "予約者数": ["予約者数", "予約人数", "予約数"],
    "契約者数": ["契約者数", "購入者数", "成約者数", "契約率計算用"],
    "契約合計金額": ["契約合計金額", "購入合計金額", "契約金額", "購入金額", "売上金額", "売上"],
    "接客対象": ["接客対象", "対象区分", "接客対象区分"],
}


def resolve_columns(df):
    normalized_actual = {norm_col(c): c for c in df.columns}
    resolved = {}
    for canonical, aliases in COLUMN_ALIASES.items():
        for alias in aliases:
            k = norm_col(alias)
            if k in normalized_actual:
                resolved[canonical] = normalized_actual[k]
                break
    return resolved


def find_detail_sheet(path):
    xls = pd.ExcelFile(path, engine="openpyxl")
    if "匿名化明細" in xls.sheet_names:
        return "匿名化明細"
    for s in xls.sheet_names:
        ns = norm_text(s)
        if "匿名" in ns and "明細" in ns:
            return s
    return None

# ============================================================
# 入力ファイル取得
# ============================================================
if INPUT_MODE.lower() == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    pattern = "**/*" if SEARCH_SUBFOLDERS else "*"
    input_paths = [
        Path(p) for p in glob.glob(str(Path(DRIVE_FOLDER) / pattern), recursive=SEARCH_SUBFOLDERS)
        if Path(p).is_file() and Path(p).suffix.lower() in [".xlsx", ".xlsm"]
    ]
else:
    uploaded = files.upload()
    input_paths = [Path(name) for name in uploaded.keys() if Path(name).suffix.lower() in [".xlsx", ".xlsm"]]

# 過去の出力を誤読込しない
input_paths = [p for p in input_paths if "予約特典別分析" not in p.name]
if not input_paths:
    raise RuntimeError("分析対象のExcelファイル（.xlsx / .xlsm）が見つかりません。")

# 完全に同一のファイルは1回だけ読む
unique_paths = []
seen_hashes = {}
for p in input_paths:
    h = sha256_file(p)
    if h not in seen_hashes:
        seen_hashes[h] = p.name
        unique_paths.append(p)

# ============================================================
# 読込・標準化
# ============================================================
frames = []
logs = []

for path in unique_paths:
    try:
        sheet = find_detail_sheet(path)
        if sheet is None:
            logs.append({"ファイル": path.name, "状態": "除外", "シート": "", "読込行数": 0,
                         "内容": "匿名化明細シートが見つからない"})
            continue

        raw = pd.read_excel(path, sheet_name=sheet, engine="openpyxl", dtype=object)
        raw = raw.dropna(how="all").copy()
        cols = resolve_columns(raw)

        required = ["来場日", "セグメント区分", "予約特典名", "接客対象"]
        missing = [c for c in required if c not in cols]
        if "契約者数" not in cols and "契約日" not in cols and "契約合計金額" not in cols:
            missing.append("契約判定列（契約者数・契約日・契約合計金額のいずれか）")
        if missing:
            logs.append({"ファイル": path.name, "状態": "除外", "シート": sheet, "読込行数": len(raw),
                         "内容": "必要列不足: " + ", ".join(missing)})
            continue

        out = pd.DataFrame(index=raw.index)
        for canonical in COLUMN_ALIASES:
            if canonical in cols:
                out[canonical] = raw[cols[canonical]]
            else:
                out[canonical] = np.nan

        out["入力ファイル"] = path.name
        out["入力シート"] = sheet

        # 予約者数：明細1行=1予約者が基本。空欄・0以下は1として扱う。
        reservation_n = to_number(out["予約者数"], default=np.nan)
        out["予約者数_集計"] = reservation_n.where(reservation_n > 0, 1).fillna(1)

        # 購入者（契約者）判定：契約者数を最優先。
        if "契約者数" in cols:
            buyer_flag = to_number(out["契約者数"], default=0).gt(0)
            buyer_rule = "契約者数>0"
        else:
            buyer_flag = pd.Series(False, index=out.index)
            if "契約日" in cols:
                buyer_flag |= is_nonblank(out["契約日"])
            if "契約合計金額" in cols:
                buyer_flag |= to_number(out["契約合計金額"], default=0).gt(0)
            buyer_rule = "契約日または契約合計金額から補完"
        out["購入者数_集計"] = buyer_flag.astype(int)

        # 来場者定義：来場日が空欄でない人 ＋ 契約者は来場者として補完。
        visit_flag = is_nonblank(out["来場日"]) | buyer_flag
        out["来場者数_集計"] = np.where(visit_flag, out["予約者数_集計"], 0)

        # 金額
        out["購入総額_集計"] = to_number(out["契約合計金額"], default=0)

        # 特典名。名称が空欄ならコード、両方空欄なら「特典不明」。
        benefit_name = out["予約特典名"].map(lambda x: unicodedata.normalize("NFKC", str(x)).strip() if not pd.isna(x) else "")
        benefit_code = out["予約特典コード"].map(norm_text)
        out["予約特典_集計"] = benefit_name.where(benefit_name.ne(""), benefit_code)
        out["予約特典_集計"] = out["予約特典_集計"].replace("", UNKNOWN_BENEFIT)

        out["接客対象_判定"] = out["接客対象"].map(norm_text).eq(norm_text(TARGET_VALUE))
        out["セグメント_判定"] = (
            out["接客対象_判定"] & out["セグメント区分"].map(norm_text).eq(norm_text(SEGMENT_VALUE))
        )

        # 同じ元データの出力ファイルを重複投入しても二重計上しないためのキー。
        id_part = out["匿名予約ID"].map(norm_text)
        venue_part = out["会場名"].map(norm_text)
        source_part = out["予約者元ファイル"].map(norm_text)
        source_part = source_part.where(source_part.ne(""), out["入力ファイル"].map(norm_text))
        out["重複判定キー"] = venue_part + "|" + source_part + "|" + id_part

        # ID欠損行は、ファイル名＋行番号で別行として保持。
        no_id = id_part.eq("")
        out.loc[no_id, "重複判定キー"] = (
            out.loc[no_id, "入力ファイル"].map(norm_text) + "|ROW|" + out.loc[no_id].index.astype(str)
        )

        frames.append(out)
        logs.append({"ファイル": path.name, "状態": "読込", "シート": sheet, "読込行数": len(out),
                     "内容": buyer_rule})

    except Exception as e:
        logs.append({"ファイル": path.name, "状態": "エラー", "シート": "", "読込行数": 0,
                     "内容": f"{type(e).__name__}: {e}"})

if not frames:
    log_df = pd.DataFrame(logs)
    display(log_df)
    raise RuntimeError("有効な明細を1件も読み込めませんでした。上の読込ログを確認してください。")

all_df = pd.concat(frames, ignore_index=True)
rows_before_dedupe = len(all_df)
all_df = all_df.drop_duplicates(subset="重複判定キー", keep="first").copy()
duplicate_rows_removed = rows_before_dedupe - len(all_df)

# 対象者全体とセグメント層
population_target = all_df[all_df["接客対象_判定"]].copy()
population_segment = all_df[all_df["セグメント_判定"]].copy()

if population_target.empty:
    raise RuntimeError(f"接客対象が『{TARGET_VALUE}』の行がありません。接客対象列の値を確認してください。")

# ============================================================
# 集計
# ============================================================
def aggregate_benefit(df, population_name):
    g = (
        df.groupby("予約特典_集計", dropna=False)
          .agg(
              予約者数=("予約者数_集計", "sum"),
              来場者数=("来場者数_集計", "sum"),
              購入者数=("購入者数_集計", "sum"),
              購入総額=("購入総額_集計", "sum"),
          )
          .reset_index()
          .rename(columns={"予約特典_集計": "予約特典"})
    )
    g["来場率"] = safe_div(g["来場者数"], g["予約者数"])
    g["契約率"] = safe_div(g["購入者数"], g["来場者数"])
    g["平均単価"] = safe_div(g["購入総額"], g["購入者数"])
    g["区分"] = population_name
    return g[["区分", "予約特典", "予約者数", "来場者数", "来場率", "購入者数", "契約率", "購入総額", "平均単価"]]


def make_total_row(detail, population_name):
    r = pd.DataFrame([{
        "区分": population_name,
        "予約特典": "【全特典計】",
        "予約者数": detail["予約者数"].sum(),
        "来場者数": detail["来場者数"].sum(),
        "購入者数": detail["購入者数"].sum(),
        "購入総額": detail["購入総額"].sum(),
    }])
    r["来場率"] = safe_div(r["来場者数"], r["予約者数"])
    r["契約率"] = safe_div(r["購入者数"], r["来場者数"])
    r["平均単価"] = safe_div(r["購入総額"], r["購入者数"])
    return r[["区分", "予約特典", "予約者数", "来場者数", "来場率", "購入者数", "契約率", "購入総額", "平均単価"]]


target_detail = aggregate_benefit(population_target, "対象者全体")
segment_detail = aggregate_benefit(population_segment, "セグメント層")

# 全体の予約者数が多い順で、全グラフの特典順を統一
order = target_detail.sort_values(["予約者数", "予約特典"], ascending=[False, True])["予約特典"].tolist()
segment_only = [x for x in segment_detail["予約特典"].tolist() if x not in order]
order += segment_only
order_map = {name: i for i, name in enumerate(order)}

def apply_order(df):
    d = df.copy()
    d["_order"] = d["予約特典"].map(order_map).fillna(999999)
    return d.sort_values(["_order", "予約特典"]).drop(columns="_order").reset_index(drop=True)

target_detail = apply_order(target_detail)
segment_detail = apply_order(segment_detail)

target_with_total = pd.concat([make_total_row(target_detail, "対象者全体"), target_detail], ignore_index=True)
segment_with_total = pd.concat([make_total_row(segment_detail, "セグメント層"), segment_detail], ignore_index=True)

# 横並び比較表
metric_cols = ["予約者数", "来場者数", "来場率", "購入者数", "契約率", "購入総額", "平均単価"]
t = target_detail.set_index("予約特典")[metric_cols].add_prefix("対象者全体_")
s = segment_detail.set_index("予約特典")[metric_cols].add_prefix("セグメント層_")
comparison = t.join(s, how="outer").reindex(order)

count_amount_cols = [c for c in comparison.columns if any(k in c for k in ["予約者数", "来場者数", "購入者数", "購入総額"])]
comparison[count_amount_cols] = comparison[count_amount_cols].fillna(0)
comparison["来場率差_pt（セグメント-対象者全体）"] = (comparison["セグメント層_来場率"] - comparison["対象者全体_来場率"]) * 100
comparison["契約率差_pt（セグメント-対象者全体）"] = (comparison["セグメント層_契約率"] - comparison["対象者全体_契約率"]) * 100
comparison["平均単価差（セグメント-対象者全体）"] = comparison["セグメント層_平均単価"] - comparison["対象者全体_平均単価"]
comparison = comparison.reset_index().rename(columns={"index": "予約特典"})

# 比較表にも全特典計を追加
def total_metrics(detail):
    reservation = detail["予約者数"].sum()
    visitor = detail["来場者数"].sum()
    buyer = detail["購入者数"].sum()
    amount = detail["購入総額"].sum()
    return {
        "予約者数": reservation,
        "来場者数": visitor,
        "来場率": visitor / reservation if reservation else np.nan,
        "購入者数": buyer,
        "契約率": buyer / visitor if visitor else np.nan,
        "購入総額": amount,
        "平均単価": amount / buyer if buyer else np.nan,
    }

tm = total_metrics(target_detail)
sm = total_metrics(segment_detail)
total_comparison = {"予約特典": "【全特典計】"}
for k, v in tm.items(): total_comparison[f"対象者全体_{k}"] = v
for k, v in sm.items(): total_comparison[f"セグメント層_{k}"] = v
total_comparison["来場率差_pt（セグメント-対象者全体）"] = (sm["来場率"] - tm["来場率"]) * 100
total_comparison["契約率差_pt（セグメント-対象者全体）"] = (sm["契約率"] - tm["契約率"]) * 100
total_comparison["平均単価差（セグメント-対象者全体）"] = sm["平均単価"] - tm["平均単価"]
comparison_with_total = pd.concat([pd.DataFrame([total_comparison]), comparison], ignore_index=True)

# ============================================================
# 可視化用データ
# ============================================================
def series_from(detail, metric):
    return detail.set_index("予約特典")[metric].reindex(order)

target_contract = series_from(target_detail, "契約率")
segment_contract = series_from(segment_detail, "契約率")
target_visit_rate = series_from(target_detail, "来場率")
segment_visit_rate = series_from(segment_detail, "来場率")
target_amount = series_from(target_detail, "購入総額").fillna(0)
segment_amount = series_from(segment_detail, "購入総額").fillna(0)
target_unit = series_from(target_detail, "平均単価")
segment_unit = series_from(segment_detail, "平均単価")

sns.set_theme(style="whitegrid", font="IPAexGothic")
plt.rcParams["axes.unicode_minus"] = False

# ---------- 1. 契約率ヒートマップ ----------
def make_rate_heatmap(metric_name, target_series, segment_series, target_num_col, target_den_col,
                      segment_num_col, segment_den_col, output_name, cbar_label):
    mat = pd.DataFrame({"対象者全体": target_series, "セグメント層": segment_series}, index=order) * 100
    mat_plot = mat.fillna(0)

    td = target_detail.set_index("予約特典")
    sd = segment_detail.set_index("予約特典")
    ann = []
    for benefit in order:
        row = []
        for detail, rate_col, num_col, den_col in [
            (td, metric_name, target_num_col, target_den_col),
            (sd, metric_name, segment_num_col, segment_den_col),
        ]:
            if benefit not in detail.index or pd.isna(detail.at[benefit, rate_col]):
                row.append("—")
            else:
                rate = detail.at[benefit, rate_col]
                num = detail.at[benefit, num_col]
                den = detail.at[benefit, den_col]
                row.append(f"{rate:.1%}\n{num:,.0f}/{den:,.0f}")
        ann.append(row)
    ann = np.array(ann)

    n = max(1, len(order))
    fig, ax = plt.subplots(figsize=(9.5, max(5.5, n * 0.52 + 2)))
    vmax = np.nanmax(mat.values) if np.isfinite(mat.values).any() else 1
    vmax = max(1, vmax)
    sns.heatmap(
        mat_plot,
        annot=ann,
        fmt="",
        cmap="YlGnBu",
        vmin=0,
        vmax=vmax,
        linewidths=1,
        linecolor="white",
        cbar_kws={"label": cbar_label},
        ax=ax,
        annot_kws={"fontsize": 9},
    )
    ax.set_title(f"予約特典別 {cbar_label}：対象者全体 vs セグメント層", fontsize=15, pad=14, weight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("予約特典")
    ax.tick_params(axis="x", rotation=0)
    ax.tick_params(axis="y", rotation=0)
    ax.text(0, -0.065, "セル内：率 ／ 分子÷分母。『—』は分母0。", transform=ax.transAxes, fontsize=9)
    plt.tight_layout()
    path = OUTPUT_DIR / output_name
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return path

contract_img = make_rate_heatmap(
    "契約率", target_contract, segment_contract,
    "購入者数", "来場者数", "購入者数", "来場者数",
    "01_予約特典別_契約率ヒートマップ.png", "契約率（%）"
)
visit_img = make_rate_heatmap(
    "来場率", target_visit_rate, segment_visit_rate,
    "来場者数", "予約者数", "来場者数", "予約者数",
    "02_予約特典別_来場率ヒートマップ.png", "来場率（%）"
)

# ---------- 2. 金額の横並び比較 ----------
def make_grouped_bar(target_series, segment_series, title, xlabel, output_name, formatter):
    n = max(1, len(order))
    y = np.arange(n)
    h = 0.36
    fig, ax = plt.subplots(figsize=(15, max(6, n * 0.52 + 2)))

    tv = target_series.reindex(order).fillna(0).values.astype(float)
    sv = segment_series.reindex(order).fillna(0).values.astype(float)

    b1 = ax.barh(y - h/2, tv, height=h, label="対象者全体", color="#4C78A8", edgecolor="black", hatch="///", linewidth=0.7)
    b2 = ax.barh(y + h/2, sv, height=h, label="セグメント層", color="#F2CF5B", edgecolor="black", hatch="...", linewidth=0.7)

    ax.set_yticks(y)
    ax.set_yticklabels(order)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=15, pad=14, weight="bold")
    ax.set_xlabel(xlabel)
    ax.legend(loc="lower right")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(yen_axis))
    ax.grid(axis="x", alpha=0.25)
    ax.grid(axis="y", visible=False)

    maxv = max(np.nanmax(tv) if len(tv) else 0, np.nanmax(sv) if len(sv) else 0, 1)
    ax.set_xlim(0, maxv * 1.30)

    labels1 = [formatter(v) if v > 0 else "" for v in tv]
    labels2 = [formatter(v) if v > 0 else "" for v in sv]
    ax.bar_label(b1, labels=labels1, padding=3, fontsize=8)
    ax.bar_label(b2, labels=labels2, padding=3, fontsize=8)

    plt.tight_layout()
    path = OUTPUT_DIR / output_name
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return path

amount_img = make_grouped_bar(
    target_amount, segment_amount,
    "予約特典別 購入総額：対象者全体 vs セグメント層",
    "購入総額（円）", "03_予約特典別_購入総額.png", yen_label
)
unit_img = make_grouped_bar(
    target_unit, segment_unit,
    "予約特典別 平均単価：対象者全体 vs セグメント層",
    "平均単価（購入総額 ÷ 購入者数）", "04_予約特典別_平均単価.png", yen_label
)

# ============================================================
# Excel出力
# ============================================================
excel_path = OUTPUT_DIR / f"予約特典別分析_{stamp}.xlsx"
log_df = pd.DataFrame(logs)

analysis_definition = pd.DataFrame([
    ["集計単位", "会場別には分けず、読み込んだ全ファイルを予約特典別に合算"],
    ["対象者全体", f"接客対象 = 『{TARGET_VALUE}』"],
    ["セグメント層", f"対象者全体のうち、セグメント区分 = 『{SEGMENT_VALUE}』"],
    ["予約者数", "予約者数列の合計。空欄・0以下は明細1行=1人として1に補完"],
    ["来場者", "来場日が空欄でない人。さらに契約者（購入者）は来場者として補完"],
    ["購入者", "契約者数 > 0 を購入者1人として集計（契約者数列がない場合のみ契約日・金額から補完）"],
    ["来場率", "来場者数 ÷ 予約者数"],
    ["契約率", "購入者数 ÷ 来場者数"],
    ["購入総額", "契約合計金額の合計"],
    ["平均単価", "購入総額 ÷ 購入者数"],
    ["重複除去", "会場名＋予約者元ファイル＋匿名予約IDが同一の行を1件に統合"],
    ["重複除去行数", duplicate_rows_removed],
    ["読込後明細行数", len(all_df)],
    ["対象者全体行数", len(population_target)],
    ["セグメント層行数", len(population_segment)],
], columns=["項目", "定義・結果"])

with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
    comparison_with_total.to_excel(writer, sheet_name="比較サマリー", index=False)
    target_with_total.to_excel(writer, sheet_name="対象者全体", index=False)
    segment_with_total.to_excel(writer, sheet_name="セグメント層", index=False)
    log_df.to_excel(writer, sheet_name="読込ログ", index=False)
    analysis_definition.to_excel(writer, sheet_name="分析定義", index=False)

    workbook = writer.book
    header_fmt = workbook.add_format({
        "bold": True, "font_color": "white", "bg_color": "#1F4E78",
        "border": 1, "align": "center", "valign": "vcenter"
    })
    total_fmt = workbook.add_format({
        "bold": True, "bg_color": "#D9EAF7", "border": 1
    })
    pct_fmt = workbook.add_format({"num_format": "0.0%", "border": 1})
    num_fmt = workbook.add_format({"num_format": "#,##0", "border": 1})
    yen_fmt = workbook.add_format({"num_format": "¥#,##0", "border": 1})
    pt_fmt = workbook.add_format({"num_format": "0.0\"pt\"", "border": 1})
    text_fmt = workbook.add_format({"border": 1})

    for sheet_name, df_out in [
        ("比較サマリー", comparison_with_total),
        ("対象者全体", target_with_total),
        ("セグメント層", segment_with_total),
        ("読込ログ", log_df),
        ("分析定義", analysis_definition),
    ]:
        ws = writer.sheets[sheet_name]
        ws.freeze_panes(1, 1)
        ws.autofilter(0, 0, max(1, len(df_out)), max(0, len(df_out.columns)-1))
        ws.set_row(0, 28)
        for c, col in enumerate(df_out.columns):
            ws.write(0, c, col, header_fmt)
            width = min(max(len(str(col)) + 3, 12), 34)
            if col in ["予約特典", "ファイル", "内容", "定義・結果"]:
                width = 42 if col in ["内容", "定義・結果"] else 28
            ws.set_column(c, c, width)

        if len(df_out) > 0:
            ws.set_row(1, None, total_fmt if sheet_name in ["比較サマリー", "対象者全体", "セグメント層"] else None)

        for c, col in enumerate(df_out.columns):
            if "率" in col and "差_pt" not in col:
                ws.set_column(c, c, 15, pct_fmt)
                if len(df_out) > 1:
                    ws.conditional_format(1, c, len(df_out), c, {
                        "type": "3_color_scale",
                        "min_color": "#FFF2CC", "mid_color": "#A9D18E", "max_color": "#4472C4"
                    })
            elif "差_pt" in col:
                ws.set_column(c, c, 20, pt_fmt)
                if len(df_out) > 1:
                    ws.conditional_format(1, c, len(df_out), c, {
                        "type": "3_color_scale",
                        "min_color": "#F4CCCC", "mid_color": "#FFFFFF", "max_color": "#B6D7A8",
                        "min_type": "min", "mid_type": "num", "mid_value": 0, "max_type": "max"
                    })
            elif any(k in col for k in ["購入総額", "平均単価", "平均単価差"]):
                ws.set_column(c, c, 18, yen_fmt)
            elif any(k in col for k in ["予約者数", "来場者数", "購入者数", "読込行数"]):
                ws.set_column(c, c, 14, num_fmt)

    # 画像は別シートに大きく配置
    image_sheets = [
        ("契約率ヒートマップ", contract_img),
        ("来場率ヒートマップ", visit_img),
        ("購入総額グラフ", amount_img),
        ("平均単価グラフ", unit_img),
    ]
    for sheet_name, image_path in image_sheets:
        ws = workbook.add_worksheet(sheet_name)
        ws.hide_gridlines(2)
        ws.set_column("A:A", 2)
        ws.insert_image("B2", str(image_path), {"x_scale": 0.72, "y_scale": 0.72})

# ============================================================
# CSVとZIP出力
# ============================================================
comparison_with_total.to_csv(OUTPUT_DIR / "比較サマリー.csv", index=False, encoding="utf-8-sig")
target_with_total.to_csv(OUTPUT_DIR / "対象者全体_特典別集計.csv", index=False, encoding="utf-8-sig")
segment_with_total.to_csv(OUTPUT_DIR / "セグメント層_特典別集計.csv", index=False, encoding="utf-8-sig")

zip_path = Path(f"/content/予約特典別分析_{stamp}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(OUTPUT_DIR))

# ============================================================
# 実行結果表示
# ============================================================
print("=" * 70)
print("分析完了")
print(f"入力ファイル数       : {len(unique_paths):,}")
print(f"読込明細行数         : {rows_before_dedupe:,}")
print(f"重複除去行数         : {duplicate_rows_removed:,}")
print(f"対象者全体           : {len(population_target):,} 行")
print(f"セグメント層         : {len(population_segment):,} 行")
print(f"出力フォルダ         : {OUTPUT_DIR}")
print(f"Excel                : {excel_path.name}")
print(f"ZIP                  : {zip_path}")
print("=" * 70)

display(comparison_with_total.head(30))

if AUTO_DOWNLOAD_ZIP:
    files.download(str(zip_path))


Saving 会場別契約率_WAF宇宮AJ43-7-1W_匿名化リスト_20260713_085513.xlsx to 会場別契約率_WAF宇宮AJ43-7-1W_匿名化リスト_20260713_085513 (1).xlsx
Saving 会場別契約率_WAF横浜AJ43-7-1W_匿名化リスト_20260713_085317.xlsx to 会場別契約率_WAF横浜AJ43-7-1W_匿名化リスト_20260713_085317 (1).xlsx
Saving 会場別契約率_WAF京AJ43-7-1W_匿名化リスト_20260713_085116.xlsx to 会場別契約率_WAF京AJ43-7-1W_匿名化リスト_20260713_085116 (1).xlsx
Saving 会場別契約率_WAF所沢AJ43-7-1W_匿名化リスト_20260713_085657.xlsx to 会場別契約率_WAF所沢AJ43-7-1W_匿名化リスト_20260713_085657 (1).xlsx
Saving 会場別契約率_WAF福井AJ43-7-1W_匿名化リスト_20260713_084950.xlsx to 会場別契約率_WAF福井AJ43-7-1W_匿名化リスト_20260713_084950 (1).xlsx
Saving 会場別契約率_WAF福津AJ43-7-1W_匿名化リスト_20260713_084735.xlsx to 会場別契約率_WAF福津AJ43-7-1W_匿名化リスト_20260713_084735 (1).xlsx
分析完了
入力ファイル数       : 6
読込明細行数         : 5,589
重複除去行数         : 0
対象者全体           : 4,295 行
セグメント層         : 1,401 行
出力フォルダ         : /content/予約特典別分析_20260713_233035
Excel                : 予約特典別分析_20260713_233035.xlsx
ZIP                  : /content/予約特典別分析_20260713_233035.zip


,予約特典,対象者全体_予約者数,対象者全体_来場者数,対象者全体_来場率,対象者全体_購入者数,対象者全体_契約率,対象者全体_購入総額,対象者全体_平均単価,セグメント層_予約者数,セグメント層_来場者数,セグメント層_来場率,セグメント層_購入者数,セグメント層_契約率,セグメント層_購入総額,セグメント層_平均単価,来場率差_pt（セグメント-対象者全体）,契約率差_pt（セグメント-対象者全体）,平均単価差（セグメント-対象者全体）
0,【全特典計】,4295,2549,0.593481,78,0.030600,52835180,677374.102564,1401,877,0.625981,60,0.068415,42114380,7.019063e+05,3.250065,3.781482,24532.230769
1,カバネリ,1765,1025,0.580737,26,0.025366,16085300,618665.384615,746,464,0.621984,22,0.047414,13332000,6.060000e+05,4.124737,2.204794,-12665.384615
2,FAIRY TAIL,575,365,0.634783,15,0.041096,8495300,566353.333333,200,132,0.660000,11,0.083333,6262300,5.693000e+05,2.521739,4.223744,2946.666667
3,まころん,525,308,0.586667,0,0.000000,0,NaN,50,33,0.660000,0,0.000000,0,NaN,7.333333,0.000000,NaN
4,ももこ,310,196,0.632258,7,0.035714,5349300,764185.714286,90,59,0.655556,7,0.118644,5349300,7.641857e+05,2.329749,8.292978,0.000000
5,カントク,251,147,0.585657,6,0.040816,4153930,692321.666667,65,36,0.553846,4,0.111111,3394930,8.487325e+05,-3.181122,7.029478,156410.833333
6,SAISAISAI,242,154,0.636364,6,0.038961,3684100,614016.666667,59,33,0.559322,2,0.060606,1772100,8.860500e+05,-7.704160,2.164502,272033.333333
7,三嶋くろね,227,137,0.603524,12,0.087591,10502250,875187.500000,87,61,0.701149,10,0.163934,9099750,9.099750e+05,9.762520,7.634319,34787.500000
8,てぃんくる,136,71,0.522059,2,0.028169,1595000,797500.000000,43,29,0.674419,1,0.034483,489500,4.895000e+05,15.235978,0.631374,-308000.000000
9,Tony,77,49,0.636364,1,0.020408,759000,759000.000000,15,10,0.666667,1,0.100000,759000,7.590000e+05,3.030303,7.959184,0.000000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>